# AI 기술면접 질문 생성기 (LoRA, 양자화 없음)

이 노트북은 **Qwen/Qwen3-1.7B** 모델을 양자화하지 않고 `bf16`으로 불러온 뒤, **LoRA 어댑터만 학습**해 지원자의 직무와 경험에 맞는 심층 기술면접 질문 3개를 생성하는 실습입니다.

핵심 흐름은 다음과 같습니다.

1. 파인튜닝 전 기본 모델이 생성하는 면접 질문을 확인합니다.
2. 직무·경험과 모범 면접 질문으로 구성된 31개 데이터를 `prompt` / `completion` 형태로 변환합니다.
3. 원본 모델 가중치는 고정하고 LoRA 어댑터만 학습합니다.
4. 학습 데이터와 직접 겹치지 않는 샘플로 파인튜닝 전/후 결과를 비교합니다.
5. 학습된 LoRA 어댑터만 파일로 저장합니다.

> 이 노트북은 **QLoRA가 아닙니다.** 모델 가중치를 4bit로 불러오지 않습니다. 다만 학습 시 VRAM을 아끼기 위해 optimizer는 `paged_adamw_8bit`를 사용합니다.


In [1]:
import os
import warnings
import logging

# 1. 파이썬 기본 경고 무시
warnings.filterwarnings("ignore")

# 2. 시스템 환경변수를 통한 Hugging Face 및 커널 로그 제어 (0=ALL, 1=INFO, 2=WARNING, 3=ERROR)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["LOGGERS_LEVEL"] = "ERROR"

# 3. transformers 자체 라이브러리 로그 레벨을 ERROR로 설정
from transformers import logging as transformers_logging
transformers_logging.set_verbosity_error()

## 0. 환경 확인

먼저 Python, PyTorch, CUDA, GPU, VRAM 상태를 확인합니다. 이 실습은 **Python 3.12 + JupyterLab** 환경을 기준으로 작성되었습니다.

- NVIDIA GPU가 정상적으로 인식되어야 합니다.
- 이 노트북의 기록된 실행 결과는 12GB VRAM GPU를 기준으로 합니다.
- VRAM이 부족하면 뒤쪽 학습 설정에서 `max_length`나 LoRA의 `r` 값을 줄이세요.


In [2]:
import torch, platform

print(f"Python 버전: {platform.python_version()}")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"총 VRAM: {total_vram:.1f} GB")
else:
    print("⚠️ GPU가 감지되지 않았습니다. NVIDIA 드라이버 / CUDA 설치를 확인하세요.")


Python 버전: 3.12.13
PyTorch 버전: 2.10.0+cu128
CUDA 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 4070 Ti
총 VRAM: 12.0 GB


## 1. 패키지 설치 및 버전 확인

처음 실행하는 환경이라면 필요한 패키지를 한 번만 설치합니다. 이미 설치되어 있다면 설치 셀은 건너뛰고, 바로 버전 확인 셀만 실행해도 됩니다.

이 노트북에서 사용하는 주요 패키지는 다음과 같습니다.

- `transformers`: Qwen 모델과 tokenizer 로드
- `peft`: LoRA 어댑터 구성 및 저장
- `trl`: `SFTTrainer`를 이용한 지도 미세조정
- `bitsandbytes`: `paged_adamw_8bit` optimizer 사용
- `datasets`: 학습 데이터셋 구성

> 여기서 `bitsandbytes`를 쓰지만, 이 노트북은 모델을 4bit로 양자화하지 않습니다. `bitsandbytes`는 optimizer 메모리 절약을 위해 사용됩니다.


In [3]:
# 최초 1회만 실행하세요. (앞의 # 을 지우고 실행)
# uv add "transformers>=5.10.1" "peft>=0.19.0" "trl>=0.24.0" "bitsandbytes>=0.48.0" \
#     "accelerate>=1.11.0" "datasets>=3.0.0" sentencepiece protobuf -U


In [4]:
import datasets

In [5]:
import transformers, peft, trl, bitsandbytes, accelerate

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [6]:
print("transformers :", transformers.__version__)
print("peft         :", peft.__version__)
print("trl          :", trl.__version__)
print("bitsandbytes :", bitsandbytes.__version__)
print("accelerate   :", accelerate.__version__)
print("datasets     :", datasets.__version__)


transformers : 5.5.0
peft         : 0.19.1
trl          : 0.24.0
bitsandbytes : 0.49.2
accelerate   : 1.14.0
datasets     : 4.3.0


## 2. 모델 로드 (양자화 없음 · bf16)

기본 모델은 **`Qwen/Qwen3-1.7B`** 입니다. 이 셀에서는 `BitsAndBytesConfig` 없이 모델을 그대로 `bf16`으로 GPU에 올립니다.

코드에서 중요한 부분은 다음과 같습니다.

- `torch_dtype=torch.bfloat16`: 모델을 bf16 정밀도로 로드합니다.
- `device_map={"": 0}`: 단일 GPU 0번에 모델을 명시적으로 올립니다.
- `tokenizer.pad_token`이 없으면 `eos_token`으로 대체합니다.

이번 실습의 목적은 양자화 기법이 아니라 **LoRA 어댑터가 직무·경험에 근거한 간결하고 구체적인 기술면접 질문 형식을 학습하는지** 확인하는 것입니다.


In [7]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

# .env 파일 로드
load_dotenv(override=True)

HF_TOKEN = os.getenv("HF_TOKEN")
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map={"": 0},
)

print("모델 로드 완료! (양자화 없음, bf16)")
if torch.cuda.is_available():
    print(f"현재 GPU 메모리 사용량: {torch.cuda.memory_allocated()/1024**3:.2f} GB")


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

모델 로드 완료! (양자화 없음, bf16)
현재 GPU 메모리 사용량: 3.20 GB


## 3. 응답 생성 함수

`ask()` 함수는 같은 지원자 정보로 파인튜닝 전과 후의 면접 질문 생성 결과를 비교하기 위한 헬퍼 함수입니다.

- `tokenizer.apply_chat_template()`으로 Qwen 채팅 형식에 맞는 입력 문장을 만듭니다.
- `enable_thinking=False`로 thinking 모드가 아니라 일반 답변 모드로 생성합니다.
- `model.eval()`을 호출해 생성 중 LoRA dropout이 켜지지 않도록 합니다.
- `torch.no_grad()`로 추론 중 gradient 계산을 끕니다.


In [ ]:
def ask(question, max_new_tokens=500):
    model.eval()  # LoRA dropout이 켜진 채 생성하면 답변이 불안정해지므로 항상 eval 모드로 고정
    messages = [{"role": "user", "content": question}] # 사용자 질문을 채팅 템플릿이 요구하는 메시지 형식으로 구성

    # 메시지를 모델이 학습한 채팅 프롬프트 형식의 문자열로 변환합니다.
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )

    # 프롬프트를 PyTorch 텐서로 토큰화한 뒤 모델이 위치한 장치로 이동합니다.
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # 추론 시에는 그래디언트 계산이 필요 없으므로 비활성화합니다.
    # 메모리 사용량이 줄고 생성 속도가 빨라집니다.
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens, # 새로 생성할 수 있는 최대 토큰 수
            do_sample=True, # 확률 분포에서 토큰을 샘플링하여 다양한 답변을 생성
            temperature=0.7, # 낮을수록 보수적이고 일관된 답변을 생성
            top_p=0.8, # 누적 확률이 top_p 이상인 토큰만 고려하여 샘플링
            repetition_penalty=1.15, # 동일한 토큰 반복을 억제
            no_repeat_ngram_size=3, # 동일한 3개 토큰 조합이 반복되지 않도록 제한
            pad_token_id=tokenizer.eos_token_id, # 별도의 PAD 토큰이 없을 경우 EOS 토큰을 패딩에 사용
        )

    # 생성된 토큰을 문자열로 변환합니다.
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    return response.strip()


## 4. Before: 파인튜닝 전 응답 확인

학습 전 기본 모델이 지원자의 직무와 경험을 얼마나 구체적으로 반영하는지 확인합니다. 생성 결과는 `before_answers`에 저장해 두고 학습 후 결과와 나란히 비교합니다.


In [10]:
test_questions = [
    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: RAG 개발자
경험: LlamaIndex와 Weaviate로 문서 검색 시스템을 구축하고 cross-encoder reranker를 적용했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: MLOps 엔지니어
경험: MLflow로 모델과 실험을 관리하고 Airflow를 이용해 재학습 파이프라인을 자동화했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 데이터 엔지니어
경험: Kafka와 Flink를 사용해 실시간 로그 처리 파이프라인을 구축하고 운영했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 백엔드 개발자
경험: Spring Boot와 Redis를 사용해 트래픽이 많은 주문 API의 응답 속도를 개선했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 컴퓨터비전 엔지니어
경험: U-Net 기반 의료영상 분할 모델을 학습하고 IoU와 Dice Score로 성능을 평가했다.""",
]

before_answers = {}
print("=" * 60)
print("파인튜닝 전(Before) 응답")
print("=" * 60)
for q in test_questions:
    ans = ask(q)
    before_answers[q] = ans
    print(f"\nQ: {q}\nA: {ans}")
    print("-" * 50)


파인튜닝 전(Before) 응답

Q: 지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: RAG 개발자
경험: LlamaIndex와 Weaviate로 문서 검색 시스템을 구축하고 cross-encoder reranker를 적용했다.
A: 다음은 **RAG(Relevant Article Generation)** 개발자가 되기 위해 필요한 실질적인 기술 면접 질문입니다. 각 질문은 지원자의 **기술적 이해도**, **실제 프로젝트 경험** 및 **해결 능력**을 평가하기 위한 목적으로 설계되었습니다.

---

### ✅ **1. 문서 검증과 관련된 문제 해결 사례**

**질문**:  
"최근에 어떤 문서 검 indexing 또는 retrieval 과정에서 발생한 문제가 있었고, 어떻게 해결했나요? 특히, 라마인드(Index)와 웨비아이브(Weaviate)의 사용 중 어떤 한계를 극복하는 방법을 설명해 주세요."

**설명**:  
이 질문은 **데이터 처리 방식**, **검색 성능**, **오류 유형** 등을 파악하며, 지원자의 기술적 지식과 문제 해결 능력을 측정합니다.

---

 ### ✅ 2. Cross-Encoder Reranker 적용 사례

**질問**:  
"LlamaIndex에서 cross-encode reranker을 적용하면서 가장 큰 도전점은 무엇이었고, 이를 해결하기 위해 어떤 전략을 했는지 설명해주세요?"

**설 명**:  
이는 **모델 선택**, **프레임워크 활용**, **성능 최적화** 등 다양한 기술 스택의 이해도를 평価합니다.

--- 

### ✆ 3. 실용적인 RAG 시스 tem 운영 환경 구성

**质问**:  
"You have built a document search system using Llama Index and Weaviace. How would you design the operational environment for this system to ensur

## 5. 데이터셋 준비

`datas/ai_interview_sft.jsonl`에 저장된 AI 기술면접 데이터 31개를 학습에 사용합니다. 각 레코드는 공통 지시문인 `instruction`, 지원자의 직무와 경험을 담은 `input`, 모범 질문 3개를 담은 `output`으로 구성됩니다. 다음 셀에서는 JSONL 파일을 Hugging Face `Dataset`으로 불러온 뒤 각 예제를 `prompt`와 `completion`으로 변환합니다.

- `prompt`: 지시문과 지원자 정보를 Qwen 채팅 템플릿으로 변환한 입력 부분
- `completion`: 모델이 생성하도록 학습할 심층 기술면접 질문 3개와 종료 토큰

뒤쪽 `SFTConfig`에서 `completion_only_loss=True`를 사용하므로 loss는 `completion` 부분에만 계산됩니다. 즉, 모델은 입력 지시문을 그대로 예측하는 대신 **직무와 경험에서 기술적 검증 포인트를 찾아 질문 3개로 구성하는 출력 패턴**을 학습합니다.


In [11]:
from datasets import load_dataset

ai_interview = load_dataset(
    "json",
    data_files="datas/ai_interview_sft.jsonl",
    split="train",
)

print(ai_interview)
print(ai_interview[1])

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 31
})
{'instruction': '지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.', 'input': '직무: LLM 엔지니어\n경험: Qwen 계열 모델을 LoRA로 파인튜닝했고 PEFT와 Transformers를 사용했다.', 'output': '1. Full Fine-Tuning 대신 LoRA를 선택한 기술적 이유를 VRAM과 trainable parameter 관점에서 설명해보세요.\n2. LoRA의 rank 값을 높이면 학습 용량과 성능에 어떤 변화가 생기나요?\n3. 학습된 adapter를 base model과 merge할 때 얻는 장점과 주의점은 무엇인가요?'}


In [12]:
# Hugging Face의 데이터셋 생성 및 처리를 위한 Dataset 클래스를 불러옵니다.
from datasets import Dataset

def format_example(example):
    """
    instruction과 input을 질문 프롬프트로 만들고,
    output을 모델이 학습할 정답으로 변환합니다.
    """
    user_content = (
        f"{example['instruction']}\n\n"
        f"{example['input']}"
    )

    # 사용자 질문을 모델 고유의 채팅 템플릿에 맞는 문자열로 변환합니다.
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_content}],
        tokenize=False, # 토큰 ID가 아닌 문자열 형태로 반환
        add_generation_prompt=True, # 모델의 답변 시작을 나타내는 프롬프트 추가
        enable_thinking=False, # 별도의 추론 과정(thinking) 출력을 비활성화
    )

    # 정답 마지막에 EOS(문장 종료) 토큰을 추가합니다.
    # 이를 통해 모델이 답변을 마쳐야 하는 시점을 학습할 수 있습니다.
    completion = example["output"] + tokenizer.eos_token

    # 프롬프트와 정답을 분리한 형태로 반환합니다.
    return {"prompt": prompt, "completion": completion}

# 데이터셋의 각 예제에 format_example 함수를 적용합니다.
# 기존 instruction, input, output 열은 유지되고, 새로 생성한 prompt, completion 열이 추가됩니다.
dataset = ai_interview.map(format_example)

print(dataset)
# 변환 결과가 올바른지 첫 번째 데이터의 프롬프트를 확인합니다.
print("PROMPT:", dataset[0]["prompt"])
# 첫 번째 데이터의 정답과 EOS 토큰 추가 여부를 확인합니다.
print("COMPLETION:", dataset[0]["completion"])


Dataset({
    features: ['instruction', 'input', 'output', 'prompt', 'completion'],
    num_rows: 31
})
PROMPT: <|im_start|>user
지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.

직무: RAG 개발자
경험: LangChain, Qdrant, BGE 임베딩을 사용해 사내 문서 QA 챗봇을 개발했다. Top-k=5를 사용했다.<|im_end|>
<|im_start|>assistant
<think>

</think>


COMPLETION: 1. Qdrant를 선택한 이유를 FAISS와 비교하여 설명해보세요.
2. Top-k=5를 결정할 때 어떤 실험이나 평가 지표를 사용했나요?
3. 임베딩 검색 결과는 관련성이 높지만 최종 답변이 부정확할 때 어느 단계를 우선 점검하겠습니까?<|im_end|>


## 6. LoRA 어댑터 설정 (양자화 없음)

이 노트북은 QLoRA가 아니므로 `prepare_model_for_kbit_training()`을 사용하지 않습니다. 대신 다음 순서로 순정 LoRA 학습을 준비합니다.

1. `model.gradient_checkpointing_enable()`로 메모리 사용량을 줄입니다.
2. `model.enable_input_require_grads()`로 gradient checkpointing 환경에서 입력 gradient를 허용합니다.
3. `LoraConfig`로 어댑터 설정을 정의합니다.
4. `get_peft_model()`로 원본 모델에 LoRA 어댑터를 붙입니다.

| 파라미터 | 의미 | 이번 실습 값 |
|---|---|---|
| `r` | LoRA rank. 클수록 표현력과 메모리 사용량이 증가합니다. | 16 |
| `lora_alpha` | LoRA scaling 계수입니다. | 32 |
| `lora_dropout` | 과적합 방지용 dropout입니다. | 0.05 |
| `target_modules` | LoRA를 붙일 attention/MLP projection layer입니다. | `q/k/v/o`, `gate/up/down` |

`model.print_trainable_parameters()` 결과에서 학습 가능한 파라미터가 0보다 커야 정상입니다. 0으로 나오면 LoRA 어댑터가 학습 대상이 아니므로, 모델 로드 셀부터 이 셀까지 순서대로 다시 실행하세요.


In [13]:
# PEFT(Parameter-Efficient Fine-Tuning) 라이브러리에서 LoRA 설정 클래스와 모델에 LoRA를 적용하는 함수를 불러옵니다.
from peft import LoraConfig, get_peft_model

# 순전파 중간 활성화 값을 모두 저장하지 않고, 역전파 시 필요한 값을 다시 계산하도록 설정합니다.
# 학습 속도가 다소 느려질 수 있지만 GPU 메모리 사용량을 줄일 수 있습니다.
model.gradient_checkpointing_enable()

# gradient checkpointing과 LoRA를 함께 사용할 때 역전파가 이어지도록 입력 임베딩 출력의 gradient 계산을 활성화합니다.
model.enable_input_require_grads()

# y = W x + (α / r) B A x
# W x              = 원본 모델의 출력
# (α / r) B A x    = LoRA adapter가 만든 보정 출력
# α / r은 LoRA 보정 출력이 원본 출력에 얼마나 강하게 반영될지 조절하는 값
# scale = (α / r)
# r : r = adapter 크기, 표현력, 병목 차원
# α : LoRA adapter의 영향력 크기를 조절하는 하이퍼파라미터
# α는 adapter의 학습 파라미터 수를 늘리는 값이 아니다. 오직 보정값의 세기를 조절한다.

# 모델에 적용할 LoRA 어댑터의 설정을 정의합니다.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,# LoRA 경로에 적용할 dropout 비율로 학습 중 과적합을 줄이는 데 도움을 줄 수 있으며, model.eval() 상태에서는 비활성화됩니다.
    bias="none", # 기존 선형 계층의 bias는 학습하지 않습니다. 따라서 주로 LoRA의 A, B 행렬만 학습됩니다
    task_type="CAUSAL_LM", # 다음 토큰을 예측하는 인과적 언어 모델임을 지정합니다.
    target_modules=[
        # LoRA 어댑터를 삽입할 선형 계층의 이름을 지정합니다.
        # Self-Attention 계층
        "q_proj", # Query 투영 계층
        "k_proj", # Key 투영 계층
        "v_proj", # Value 투영 계층
        "o_proj", # Attention 출력 투영 계층

        # MLP 또는 Feed-Forward 계층
        "gate_proj", # 게이트 투영 계층
        "up_proj", # 중간 차원을 확장하는 투영 계층
        "down_proj", # 확장된 차원을 다시 줄이는 투영 계층
    ],
)

# 기존 모델의 지정된 target_modules에 LoRA 어댑터를 삽입합니다.
# 반환된 모델은 원본 가중치 대부분이 고정되고,
# LoRA 어댑터 파라미터를 중심으로 학습되는 PEFT 모델입니다.
model = get_peft_model(model, lora_config)

# 전체 파라미터 수, 학습 가능한 파라미터 수와 비율을 출력하여 LoRA가 정상적으로 적용되었는지 확인합니다.
model.print_trainable_parameters()


trainable params: 17,432,576 || all params: 1,738,007,552 || trainable%: 1.0030


## 7. 학습 설정 및 실행

`SFTTrainer`로 지원자의 직무와 경험에 맞는 심층 기술면접 질문을 생성하도록 지도 미세조정합니다. 작은 데이터셋으로 질문의 형식과 구체성을 학습하는 실습이므로 전체 모델이 아니라 LoRA 어댑터만 업데이트합니다.

주요 설정은 다음과 같습니다.

- `per_device_train_batch_size=1`: GPU 메모리를 아끼기 위해 실제 배치는 1로 둡니다.
- `gradient_accumulation_steps=8`: 8번의 mini-batch를 모아 한 번 업데이트합니다.
- `learning_rate=2e-4`: LoRA 학습에서 자주 쓰는 범위의 학습률입니다.
- `bf16=True`: bf16 연산을 사용합니다.
- `optim="paged_adamw_8bit"`: optimizer 상태 메모리를 절약합니다.
- `max_length=512`: 입력과 출력 토큰을 합친 최대 길이입니다.
- `completion_only_loss=True`: 질문이 아니라 assistant 답변 부분 위주로 loss를 계산합니다.

학습이 끝나면 `model.eval()`로 전환합니다. LoRA dropout이 켜진 train mode에서 생성하면 답변이 불안정해질 수 있기 때문입니다.

> 오류 점검: loss가 전혀 줄지 않거나 학습이 되는 것처럼 보이는데 결과가 변하지 않으면, `model.print_trainable_parameters()`에서 trainable parameter가 0이 아닌지 먼저 확인하세요.


In [14]:
# TRL 라이브러리에서 지도 미세 조정(SFT)에 사용하는 Trainer 클래스와 학습 설정 클래스를 불러옵니다.
from trl import SFTTrainer, SFTConfig

# SFT(Supervised Fine-Tuning) 학습에 사용할 설정을 정의합니다.
training_args = SFTConfig(
    output_dir="./output/sample_ai_interview", # 체크포인트와 학습 결과가 저장될 디렉터리
    num_train_epochs=8, # 전체 학습 데이터셋을 반복해서 학습할 횟수
    per_device_train_batch_size=1, # GPU 한 장이 한 번의 순전파·역전파에서 처리할 데이터 수
    gradient_accumulation_steps=8,  # 8개 미니 배치의 gradient를 누적한 뒤 한 번 가중치를 업데이트합니다. # GPU가 1개라면 실질적인 배치 크기는 1 × 8 = 8입니다.
    gradient_checkpointing=True, # 중간 활성화 값을 저장하는 대신 역전파 시 다시 계산하여 GPU 메모리 사용량을 줄입니다. 대신 학습 시간이 다소 증가합니다.
    learning_rate=2e-4, # 옵티마이저가 학습 파라미터를 한 번에 얼마나 변경할지 결정합니다. LoRA 학습에서 흔히 사용되는 비교적 높은 학습률입니다.
    logging_steps=5, # 5번의 학습 스텝마다 loss 등의 학습 상태를 출력합니다.
    save_strategy="no", # 학습 도중 체크포인트를 자동으로 저장하지 않습니다. 따라서 학습 후 어댑터가 필요하면 별도로 save_pretrained()를 호출해야 합니다.
    bf16=True, # bfloat16 정밀도를 사용하여 메모리 사용량을 줄이고 학습 속도를 높입니다. 사용 중인 GPU가 BF16 연산을 지원해야 합니다.
    optim="paged_adamw_8bit", # 8비트 paged AdamW 옵티마이저를 사용하여 일반 AdamW보다 옵티마이저 상태가 차지하는 GPU 메모리를 줄입니다.
    max_length=512, # 하나의 학습 예제가 가질 수 있는 최대 토큰 길이로, 이를 초과하는 입력은 잘릴 수 있습니다.
    completion_only_loss=True, # prompt 부분은 loss 계산에서 제외하고 completion, 즉 정답 부분에 대해서만 loss 계산에 포함합니다.
    report_to="none", # Weights & Biases 등의 외부 실험 추적 서비스에 로그를 전송하지 않습니다.
)
# 설정한 모델, 학습 옵션, 데이터셋을 사용하여 지도 미세 조정용 Trainer를 생성합니다.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# 실제 SFT 학습을 시작합니다.
# completion_only_loss=True이므로 정답 부분을 중심으로 학습합니다.
trainer.train()

# 학습이 끝나면 반드시 eval 모드로 전환합니다.
# (LoRA dropout이 생성 중에도 계속 켜져 있으면 답변이 불안정해집니다)
model.eval()
print("학습 완료! model.eval() 적용됨 — 이제 안정적으로 답변을 생성할 수 있습니다.")


{'loss': '2.559', 'grad_norm': '0.162', 'learning_rate': '0.000175', 'entropy': '1.664', 'num_tokens': '6372', 'mean_token_accuracy': '0.5542', 'epoch': '1.258'}
{'loss': '1.756', 'grad_norm': '0.1406', 'learning_rate': '0.0001437', 'entropy': '2.344', 'num_tokens': '1.273e+04', 'mean_token_accuracy': '0.6254', 'epoch': '2.516'}
{'loss': '1.501', 'grad_norm': '0.1259', 'learning_rate': '0.0001125', 'entropy': '2.257', 'num_tokens': '1.924e+04', 'mean_token_accuracy': '0.6816', 'epoch': '3.774'}
{'loss': '1.311', 'grad_norm': '0.1342', 'learning_rate': '8.125e-05', 'entropy': '2.009', 'num_tokens': '2.542e+04', 'mean_token_accuracy': '0.715', 'epoch': '5'}
{'loss': '1.138', 'grad_norm': '0.1378', 'learning_rate': '5e-05', 'entropy': '1.783', 'num_tokens': '3.178e+04', 'mean_token_accuracy': '0.7446', 'epoch': '6.258'}
{'loss': '1.07', 'grad_norm': '0.1376', 'learning_rate': '1.875e-05', 'entropy': '1.699', 'num_tokens': '3.822e+04', 'mean_token_accuracy': '0.7576', 'epoch': '7.516'}
{'t

## 8. After: 파인튜닝 후 응답 비교

학습 전에 저장해 둔 `before_answers`와 학습 후 `ask()` 결과를 나란히 출력합니다. 학습 데이터에 없던 직무·기술 조합에서도 정확히 3개의 질문을 생성하는지, 지원자의 경험에 나온 기술을 구체적으로 활용하는지, 무관한 기술을 임의로 끌어오지 않는지 확인합니다.


In [15]:
test_questions = [
    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: RAG 개발자
경험: LlamaIndex와 Weaviate로 문서 검색 시스템을 구축하고 cross-encoder reranker를 적용했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: MLOps 엔지니어
경험: MLflow로 모델과 실험을 관리하고 Airflow를 이용해 재학습 파이프라인을 자동화했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 데이터 엔지니어
경험: Kafka와 Flink를 사용해 실시간 로그 처리 파이프라인을 구축하고 운영했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 백엔드 개발자
경험: Spring Boot와 Redis를 사용해 트래픽이 많은 주문 API의 응답 속도를 개선했다.""",

    """지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: 컴퓨터비전 엔지니어
경험: U-Net 기반 의료영상 분할 모델을 학습하고 IoU와 Dice Score로 성능을 평가했다.""",
]

print("=" * 60)
print("AI 기술면접 질문 Before/After 비교")
print("=" * 60)
for q in test_questions:
    after_ans = ask(q)
    print(f"\nQ: {q}")
    print(f"[Before] {before_answers[q]}")
    print(f"[After ] {after_ans}")
    print("-" * 50)


interview Q&A

Q: 지원자의 경험을 바탕으로 실제 역량을 검증할 수 있는 심층 기술면접 질문 3개를 만들어라.
직무: RAG 개발자
경험: LlamaIndex와 Weaviate로 문서 검색 시스템을 구축하고 cross-encoder reranker를 적용했다.
[Before] 다음은 **RAG(Relevant Article Generation)** 개발자가 되기 위해 필요한 실질적인 기술 면접 질문입니다. 각 질문은 지원자의 **기술적 이해도**, **실제 프로젝트 경험** 및 **해결 능력**을 평가하기 위한 목적으로 설계되었습니다.

---

### ✅ **1. 문서 검증과 관련된 문제 해결 사례**

**질문**:  
"최근에 어떤 문서 검 indexing 또는 retrieval 과정에서 발생한 문제가 있었고, 어떻게 해결했나요? 특히, 라마인드(Index)와 웨비아이브(Weaviate)의 사용 중 어떤 한계를 극복하는 방법을 설명해 주세요."

**설명**:  
이 질문은 **데이터 처리 방식**, **검색 성능**, **오류 유형** 등을 파악하며, 지원자의 기술적 지식과 문제 해결 능력을 측정합니다.

---

 ### ✅ 2. Cross-Encoder Reranker 적용 사례

**질問**:  
"LlamaIndex에서 cross-encode reranker을 적용하면서 가장 큰 도전점은 무엇이었고, 이를 해결하기 위해 어떤 전략을 했는지 설명해주세요?"

**설 명**:  
이는 **모델 선택**, **프레임워크 활용**, **성능 최적화** 등 다양한 기술 스택의 이해도를 평価합니다.

--- 

### ✆ 3. 실용적인 RAG 시스 tem 운영 환경 구성

**质问**:  
"You have built a document search system using Llama Index and Weaviace. How would you design the operational environment for this system to ens

## 9. LoRA 어댑터 저장

이 셀은 전체 모델이 아니라 **LoRA 어댑터만** 저장합니다. 저장 위치는 다음과 같습니다.

```text
./lora_adapters/sample_ai_interview
```

어댑터만 저장하면 파일 크기가 작고, 나중에 같은 base model에 다시 붙여서 사용할 수 있습니다. 전체 모델로 배포하려면 별도의 병합 과정이 필요합니다.


In [16]:
ADAPTER_DIR = "./lora_adapters/sample_ai_interview"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA 어댑터 저장 완료: {ADAPTER_DIR}")


LoRA 어댑터 저장 완료: ./lora_adapters/sample_ai_interview


## 정리

이 노트북에서 확인한 내용은 다음과 같습니다.

- 양자화 없이 `Qwen/Qwen3-1.7B`를 bf16으로 로드했습니다.
- 원본 모델 전체를 학습하지 않고 LoRA 어댑터만 학습했습니다.
- `instruction` / `input` / `output` 데이터를 `prompt` / `completion` 학습 형식으로 변환했습니다.
- `completion_only_loss=True`를 사용해 모범 면접 질문 부분에만 loss를 계산했습니다.
- 학습 데이터와 직접 겹치지 않는 지원자 사례로 파인튜닝 전/후 질문의 구체성과 형식을 비교했습니다.
- 최종 산출물로 AI 기술면접 질문 생성용 LoRA 어댑터를 저장했습니다.

다음 단계로는 저장된 어댑터를 다시 불러오거나, base model과 병합한 뒤 Ollama 같은 로컬 실행 환경에 배포할 수 있습니다.
